# Bài 3: Bảo vệ Dữ liệu và "Phép thuật" với Special Methods

## Mục tiêu bài học

Sau bài học này, bạn sẽ có thể:
- Hiểu rõ khái niệm **Tính đóng gói (Encapsulation)** và tầm quan trọng của nó.
- Phân biệt và sử dụng các quy ước về thuộc tính: `public`, `_protected`, và `__private`.
- Sử dụng **Properties (`@property`)** để tạo các getter và setter an toàn.
- Hiểu và triển khai các **Phương thức đặc biệt (Special/Dunder Methods)** như `__str__`, `__repr__`, và `__len__`.
- Làm cho các object của bạn hoạt động một cách tự nhiên và "Pythonic" hơn.

## 1. Tính đóng gói (Encapsulation)

Tính đóng gói là một trong bốn trụ cột của OOP. Ý tưởng cốt lõi của nó là **che giấu thông tin (information hiding)**. Điều này có nghĩa là dữ liệu (thuộc tính) của một object không nên được truy cập và thay đổi một cách tự do từ bên ngoài. Thay vào đó, object nên cung cấp các phương thức công khai (public methods) để thế giới bên ngoài tương tác với nó.

**Tại sao phải làm vậy?**

1.  **Bảo vệ dữ liệu:** Ngăn chặn việc gán các giá trị không hợp lệ cho thuộc tính. Ví dụ: tuổi không thể là số âm, email phải có định dạng đúng.
2.  **Dễ thay đổi và bảo trì:** Nếu bạn muốn thay đổi cách lưu trữ dữ liệu bên trong (ví dụ: từ một biến thành một list), bạn chỉ cần thay đổi logic bên trong class. Code bên ngoài sử dụng class đó không cần phải thay đổi vì nó chỉ tương tác qua các public method.
3.  **Kiểm soát truy cập:** Bạn có thể quyết định một thuộc tính chỉ có thể đọc (read-only) chứ không thể ghi.

### Vấn đề của việc truy cập trực tiếp

Hãy xem lại class `Dog` của chúng ta. Hiện tại, không có gì ngăn cản chúng ta gán một giá trị vô lý cho tuổi của chó.

In [ ]:
class Dog:
    def __init__(self, name, age):
        self.name = name
        self.age = age

my_dog = Dog("Milo", 3)
print(f"{my_dog.name} đang {my_dog.age} tuổi.")

# Gán một giá trị không hợp lệ
my_dog.age = -5
print(f"Tuổi mới của {my_dog.name}: {my_dog.age}") # Tuổi âm, thật vô lý!

### Quy ước đặt tên trong Python

Không giống như các ngôn ngữ khác (Java, C++), Python không có từ khóa `private` hay `public` để ép buộc việc che giấu dữ liệu. Thay vào đó, Python sử dụng các **quy ước đặt tên**.

- **`name` (Public):** Thuộc tính/phương thức có thể được truy cập tự do từ bất cứ đâu. Đây là mặc định.
- **`_name` (Protected):** Một dấu gạch dưới đứng trước. Đây là một **gợi ý** cho lập trình viên khác rằng "đây là thuộc tính/phương thức nội bộ, đừng truy cập trực tiếp từ bên ngoài trừ khi bạn biết rõ mình đang làm gì". Về mặt kỹ thuật, bạn vẫn có thể truy cập nó.
- **`__name` (Private):** Hai dấu gạch dưới đứng trước. Đây là cơ chế mạnh hơn. Python sẽ tự động "bóp méo" tên của thuộc tính này (gọi là *name mangling*) để làm cho việc truy cập từ bên ngoài khó hơn rất nhiều (nhưng không phải là không thể). Ví dụ, `__age` sẽ bị đổi thành `_ClassName__age`.

In [ ]:
class MyClass:
    def __init__(self):
        self.public_var = "I am public"
        self._protected_var = "I am protected"
        self.__private_var = "I am private"

obj = MyClass()

print(obj.public_var)
print(obj._protected_var) # Vẫn truy cập được, nhưng đây là hành động không được khuyến khích

try:
    print(obj.__private_var)
except AttributeError as e:
    print(f"Lỗi: {e}")

# Vẫn có thể truy cập nếu biết tên đã bị bóp méo
print("Truy cập bằng name mangling:", obj._MyClass__private_var)

**Quy tắc chung:** Hãy bắt đầu với các thuộc tính public. Nếu bạn cần bảo vệ nó, hãy chuyển sang `_protected`. Chỉ sử dụng `__private` khi bạn có lý do thực sự mạnh mẽ để tránh xung đột tên trong kế thừa.

### 2. Properties: Getter và Setter kiểu Pythonic

Vậy làm thế nào để kiểm soát việc truy cập một cách thanh lịch? Sử dụng **properties**.

Một property cho phép bạn "bọc" một thuộc tính nội bộ (ví dụ `_age`) bằng các phương thức getter (để lấy giá trị) và setter (để gán giá trị), nhưng từ bên ngoài, người dùng vẫn truy cập nó như một thuộc tính bình thường.

- **`@property`**: Biến một phương thức thành một "getter". Phương thức này sẽ được gọi khi bạn truy cập thuộc tính.
- **`@<tên_thuộc_tính>.setter`**: Biến một phương thức thành một "setter". Phương thức này sẽ được gọi khi bạn gán giá trị cho thuộc tính.

In [ ]:
class Person:
    def __init__(self, name, age):
        self.name = name
        # Gán giá trị cho age thông qua setter ngay trong constructor
        self.age = age 

    @property
    def age(self):
        """Đây là getter. Nó được gọi khi ta truy cập `person.age`."""
        print("(Getting age)")
        return self._age

    @age.setter
    def age(self, value):
        """Đây là setter. Nó được gọi khi ta gán `person.age = value`."""
        print(f"(Setting age to {value})")
        if value < 0:
            raise ValueError("Tuổi không thể là số âm.")
        self._age = value # Lưu giá trị vào thuộc tính "protected"

# Tạo object
p = Person("Alice", 30)

# Truy cập age (sẽ gọi getter)
print(f"Tuổi của {p.name} là {p.age}")

# Gán giá trị mới cho age (sẽ gọi setter)
p.age = 31

# Thử gán giá trị không hợp lệ
try:
    p.age = -10
except ValueError as e:
    print(f"Lỗi khi gán tuổi: {e}")

print(f"Tuổi cuối cùng: {p.age}")

Với property, chúng ta đã bảo vệ được thuộc tính `age` một cách hiệu quả mà vẫn giữ được cú pháp truy cập `p.age` đơn giản và đẹp đẽ.

## 3. Phương thức đặc biệt (Special/Dunder Methods)

Bạn có bao giờ tự hỏi tại sao `len([1, 2, 3])` lại hoạt động? Hoặc tại sao `print(my_list)` lại cho ra một chuỗi đẹp đẽ? Đó là nhờ các phương thức đặc biệt, hay còn gọi là **dunder methods** (vì chúng có hai dấu gạch dưới ở đầu và cuối, **d**ouble **under**score).

Những phương thức này cho phép object của chúng ta tích hợp với các hàm và toán tử sẵn có của Python.

### `__str__` và `__repr__`

Đây là hai phương thức quan trọng nhất để biểu diễn object dưới dạng chuỗi.

- **`__str__(self)`**: Được gọi bởi hàm `str(object)` và `print(object)`. Mục tiêu là trả về một chuỗi **thân thiện với người dùng**, dễ đọc.
- **`__repr__(self)`**: Được gọi bởi hàm `repr(object)` và được hiển thị khi bạn gõ tên object trong một interactive console. Mục tiêu là trả về một chuỗi **rõ ràng, không mơ hồ**, lý tưởng nhất là một chuỗi mà khi copy-paste vào code Python sẽ tạo lại được object đó.

**Quy tắc:** Luôn luôn nên implement `__repr__`. Nếu bạn không implement `__str__`, Python sẽ dùng `__repr__` thay thế.

In [ ]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __str__(self):
        # Dành cho người dùng cuối
        return f"{self.name}, {self.age} tuổi"

    def __repr__(self):
        # Dành cho lập trình viên
        return f"Person(name='{self.name}', age={self.age})"

p = Person("Bob", 42)

# print() sẽ gọi __str__
print(p)

# str() sẽ gọi __str__
print(str(p))

# repr() sẽ gọi __repr__
print(repr(p))

# Trong interactive console (như Jupyter), gõ tên biến sẽ gọi __repr__
p

### `__len__`

Cho phép object của bạn hoạt động với hàm `len()`.

## Ví dụ trong Khoa học dữ liệu: Nâng cấp Class `Dataset`

Bây giờ, hãy áp dụng tất cả những kiến thức này để nâng cấp class `SimpleDataset` từ bài 1. Chúng ta sẽ làm cho nó an toàn và "thông minh" hơn.

In [ ]:
import pandas as pd

class SmartDataset:
    """Một class dataset được cải tiến với encapsulation và special methods."""
    
    def __init__(self, data, name):
        # Sử dụng setter để kiểm tra kiểu dữ liệu ngay khi khởi tạo
        self.data = data 
        self.name = name

    @property
    def data(self):
        """Getter cho thuộc tính data."""
        return self._data

    @data.setter
    def data(self, new_data):
        """Setter cho thuộc tính data, đảm bảo nó luôn là DataFrame."""
        if not isinstance(new_data, pd.DataFrame):
            raise TypeError("Dữ liệu phải là một pandas DataFrame.")
        self._data = new_data

    # --- Special Methods ---
    
    def __str__(self):
        return f"Dataset '{self.name}' với kích thước {self.data.shape}"
    
    def __repr__(self):
        return f"SmartDataset(name='{self.name}', shape={self.data.shape})"
    
    def __len__(self):
        """Trả về số dòng của dataset."""
        return len(self.data)
    
    def __getitem__(self, key):
        """Cho phép truy cập dữ liệu bằng slicing hoặc tên cột."""
        print(f"(Truy cập vào dataset với key: {key})")
        return self.data[key]

    # --- Các method tiện ích khác ---
    
    def summary(self):
        return self.data.describe()

Hãy xem class mới của chúng ta hoạt động như thế nào.

In [ ]:
sample_df = pd.DataFrame({
    'A': [1, 2, 3, 4],
    'B': [5, 6, 7, 8]
})

sds = SmartDataset(sample_df, "MyData")

# 1. __str__ được gọi
print(sds)

# 2. __repr__ được gọi
sds

In [ ]:
# 3. __len__ được gọi
print(f"Số dòng trong dataset: {len(sds)}")

In [ ]:
# 4. __getitem__ được gọi
# Lấy một cột
print("\nCột A:")
print(sds['A'])

# Lấy các dòng đầu tiên
print("\nCác dòng đầu:")
print(sds[0:2])

In [ ]:
# 5. Setter được gọi để kiểm tra kiểu dữ liệu
try:
    # Thử gán một list thay vì DataFrame
    sds.data = [1, 2, 3]
except TypeError as e:
    print(f"Lỗi: {e}")

Bằng cách sử dụng các kỹ thuật này, class `SmartDataset` của chúng ta đã trở nên an toàn hơn, dễ sử dụng hơn và tích hợp liền mạch với các tính năng của Python.

## Tóm tắt

- **Tính đóng gói (Encapsulation)** là việc che giấu dữ liệu nội bộ và chỉ cung cấp các phương thức public để tương tác, giúp bảo vệ dữ liệu và dễ bảo trì.
- Python dùng quy ước đặt tên: `_protected` (gợi ý không nên dùng) và `__private` (bị bóp méo tên).
- **`@property`** và **`@<name>.setter`** là cách Pythonic để tạo getter và setter, cho phép kiểm soát truy cập mà vẫn giữ cú pháp đẹp.
- **Special/Dunder Methods** (`__tên__`) cho phép object của bạn hoạt động với các hàm và toán tử của Python (như `len()`, `print()`, `[]`).
- `__str__` cho người dùng, `__repr__` cho lập trình viên.
- Việc áp dụng các nguyên tắc này giúp code của bạn trở nên chuyên nghiệp, an toàn và dễ sử dụng hơn.

## Bài học tiếp theo

Chúng ta đã nắm vững các khối xây dựng cơ bản của OOP. Trong bài học tiếp theo, chúng ta sẽ xem xét cách kết hợp chúng lại với nhau để tạo ra các **Mẫu thiết kế (Design Patterns)** hữu ích cho các quy trình khoa học dữ liệu, giúp giải quyết các vấn đề phổ biến một cách hiệu quả.